# Short Root Cause Audit

Notebook metodológico principal del bloque `short`.

Usa `FINRA` como baseline oficial y `Polygon` como provider secundario de comparación.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

CACHE_ROOT = Path(r"C:\TSIS_Data\v1\backtest_SmallCaps\notebooks\00_data_certification\auditoria\short\cache_v2")
NOTEBOOK_ROOT = CACHE_ROOT.parent
plt.style.use('default')
sns.set_theme(style='whitegrid')

In [ ]:
%run C:/TSIS_Data/02_backtest_SmallCaps/notebooks/00_data_certification/auditoria/short/cell_code/00_load_short_audit_artifacts.py

## 1. Snapshot Ejecutivo

In [ ]:
manifest = json.loads((CACHE_ROOT / 'short_build_manifest.json').read_text(encoding='utf-8'))
display(pd.DataFrame([manifest]))
display(Markdown(f"**Artifacts en cache_v2:** `{len(list(CACHE_ROOT.glob('*.parquet')))} parquet(s)`"))

## 2. FINRA vs Polygon

In [ ]:
provider_counts = (
    short_provider_inventory.groupby(['dataset','provider'], as_index=False)
    .agg(files=('ticker','size'), nonzero_files=('rows', lambda s: int((s > 0).sum())), min_date=('min_date','min'), max_date=('max_date','max'))
)
display(provider_counts)

comparison_counts = (
    short_provider_comparison_summary.groupby('dataset', as_index=False)
    .agg(only_polygon=('only_polygon','sum'), only_finra=('only_finra','sum'), shared=('shared','sum'))
)
display(comparison_counts)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
sns.barplot(data=provider_counts, x='dataset', y='nonzero_files', hue='provider', ax=ax[0])
ax[0].set_title('Files con filas > 0')
sns.barplot(data=comparison_counts.melt(id_vars='dataset', var_name='bucket', value_name='count'), x='dataset', y='count', hue='bucket', ax=ax[1])
ax[1].set_title('Comparison bucket')
plt.tight_layout()
plt.show()

## 3. Integridad Aritmética

In [ ]:
si_summary = short_interest_arithmetic_checks.groupby('provider', as_index=False).agg(
    files=('ticker','size'),
    zero_adv_rows=('zero_adv_rows','sum'),
    mean_abs_err=('days_to_cover_abs_err_mean','mean'),
    p95_abs_err=('days_to_cover_abs_err_p95','mean'),
    max_abs_err=('days_to_cover_abs_err_max','max'),
)
sv_summary = short_volume_arithmetic_checks.groupby('provider', as_index=False).agg(
    files=('ticker','size'),
    mean_ratio_err=('ratio_err_mean','mean'),
    p95_ratio_err=('ratio_err_p95','mean'),
    mean_venue_err=('venue_err_mean','mean'),
    p95_venue_err=('venue_err_p95','mean'),
)
display(Markdown('**short_interest**'))
display(si_summary)
display(Markdown('**short_volume**'))
display(sv_summary)

## 4. Identidad y Residuo Only Polygon

In [ ]:
identity_counts = short_identity_links['identity_link_bucket'].value_counts(dropna=False).rename_axis('bucket').reset_index(name='count')
display(identity_counts)
display(short_identity_links[['ticker','dataset','poly_rows','poly_min_date','poly_max_date','identity_link_bucket','ticker_base_mode','identity_bucket_mode']].head(30))

In [ ]:
fig, ax = plt.subplots(figsize=(8,4))
sns.barplot(data=identity_counts, y='bucket', x='count', ax=ax)
ax.set_title('Only Polygon identity buckets')
plt.tight_layout()
plt.show()

## 5. Causalidad Short Volume

In [ ]:
display(short_causal_alignment_summary)
display(short_volume_halt_link_candidates[['ticker','date','short_volume_ratio','short_signal_bucket','market_link_bucket','halt_visual_bucket','quotes_severity','trades_severity']].head(30))

In [ ]:
fig, ax = plt.subplots(figsize=(8,4))
sns.barplot(data=short_causal_alignment_summary, x='market_link_bucket', y='cases', ax=ax)
ax.set_title('short_volume causal buckets')
ax.tick_params(axis='x', rotation=20)
plt.tight_layout()
plt.show()

## 6. Contexto Short Interest

In [ ]:
display(short_interest_context_summary)
display(short_interest_market_context_candidates[['ticker','settlement_date','days_to_cover','dtc_z','linked_halt_date','days_from_halt','halt_visual_bucket','context_bucket']].head(30))

## 7. Lectura Actual

- `short_volume` gana valor real como contexto de anomalia de mercado, más que como explicador exclusivo de halts.
- `short_interest` aporta una capa lenta de crowding / squeeze risk cerca de algunos halts.
- `FINRA` queda fijado como baseline oficial.
- `Polygon` queda como capa comparativa secundaria.

## 8. Viewer Causal

Usa este viewer para revisar manualmente casos de `short_volume` y `short_interest` y calibrar la pol?tica `good / review / bad`.


In [ ]:
get_ipython().run_line_magic('run', r'C:/TSIS_Data/02_backtest_SmallCaps/notebooks/00_data_certification/auditoria/short/cell_code/01_short_causal_overlay.py')
build_short_viewer()
